## Heudiconv wrapper
Runs heudiconv (no conversion) on all sessions to get a `dicominfo.tsv` per session, then removes sensitive columns.

Output: `<out_dir>/.heudiconv/<session>/info/dicominfo_mod-<session>.tsv` (`out_dir` is printed by `cfg.show()`)

In [ ]:
from seqinspect import Config, load_config

# Pick ONE way to configure (A is the default, the file is gitignored):
# A) config file
cfg = load_config("configs/STAD_example.py")

# B) config file, but with a different root
# cfg = load_config("configs/myproject.py", root="/path/to/labdrive")

# C) quick inline config for another dataset (careful: the pre-commit hook blocks absolute paths)
# cfg = Config(
#     identifier="quicklook",
#     dicom_root="/path/to/dicoms",
#     dcm_pattern="/path/to/dicoms/{subject}/SCANS/*/DICOM/*.dcm",
#     modalities={"T1w": {"include": ["t1"]}, "fMRI": {"include": ["bold"], "exclude": ["SBRef"]}},
# )

# D) MRE project: one config per scanner, the drive root is found automatically
# cfg = load_config("configs/mre_7T.py")   # or "configs/mre_3T.py"

cfg.show()

In [ ]:
from seqinspect import find_sessions, run_heudiconv, clean_dicominfo, check_consistency, check_patient_names

sessions = find_sessions(cfg)
print(f"{len(sessions)} sessions found")
sessions

In [ ]:
# only some sessions, e.g. for testing
# sessions = [sessions[0]]

### Optional: folder names vs. dicom patient_name
Apart from naming conflicts (note them here), folders should be renamed to match the patient_name. Reads one dicom per session (needs pydicom).

In [ ]:
names = check_patient_names(cfg, sessions)
names[~names["is match?"]]

In [ ]:
FORCE = False   # True: re-run heudiconv even if a dicominfo file exists
DRYRUN = False  # True: only print the heudiconv commands
run_heudiconv(cfg, sessions, force=FORCE, dryrun=DRYRUN)

### Remove sensitive columns

In [ ]:
clean_dicominfo(cfg)

### Check for missing data or info sheets

In [ ]:
check_consistency(cfg, sessions)